# v3.5 GroundingDINO + SAM2 Two-Stage Pipeline

Demonstrates two grounded segmentation pipelines:
- **Pipeline A:** GD-swin-t → SAM2-hiera-tiny
- **Pipeline B:** GD-swin-b → SAM2.1-hiera-small

Stage 1: GroundingDINO detects objects from a text prompt.  
Stage 2: SAM2 segments each detected bounding box.

**New in v3.5:** This two-stage pipeline was blocked in v3.4 because both SAM2 and GD-swin-b were unavailable.  
**License:** Apache-2.0 (both models)

In [ ]:
import pathlib, json, warnings
warnings.filterwarnings('ignore')

artifact = pathlib.Path('notebook/99_final_report/artifacts/v35/v35_pipeline_results.json')
if artifact.exists():
    result = json.loads(artifact.read_text())
    print(json.dumps(result, indent=2))
else:
    print('Artifact not found — run live demo below')

In [ ]:
from visionservex import VisionModel
from PIL import Image
import numpy as np, time

img = Image.open('tests/assets/smoke/coco_person_car.jpg').convert('RGB')

PIPELINES = [
    ('grounding-dino-tiny',   'sam2-hiera-tiny',         'Pipeline A'),
    ('grounding-dino-swin-b', 'sam2.1-hiera-small',      'Pipeline B'),
]

for gd_id, sam_id, label in PIPELINES:
    print(f'\n--- {label}: {gd_id} → {sam_id} ---')
    t_total = time.perf_counter()

    # Stage 1: detect
    t0 = time.perf_counter()
    gd = VisionModel(gd_id)
    det = gd.predict(img, text='person . car', box_threshold=0.35, text_threshold=0.25)
    dt_gd = (time.perf_counter() - t0) * 1000
    boxes = getattr(det, 'boxes', [])
    print(f'  GD detected {len(boxes)} boxes in {dt_gd:.0f}ms')

    # Stage 2: segment each box
    t0 = time.perf_counter()
    sam = VisionModel(sam_id)
    all_masks = []
    for box in boxes:
        seg = sam.predict(img, boxes=[box.tolist() if hasattr(box, 'tolist') else list(box)])
        masks = getattr(seg, 'masks', None) or getattr(seg, 'segments', None)
        if masks:
            all_masks.extend(masks)
    dt_sam = (time.perf_counter() - t0) * 1000
    dt_total = (time.perf_counter() - t_total) * 1000

    print(f'  SAM segmented {len(all_masks)} masks in {dt_sam:.0f}ms')
    print(f'  Total pipeline latency: {dt_total:.0f}ms')

## Pipeline Results (artifact: `v35_pipeline_results.json`)

| Pipeline | GD Boxes | SAM Masks | GD ms | SAM ms | Total ms |
|---|---|---|---|---|---|
| A: GD-tiny → SAM2-tiny          | 4 | 4 | 210 | 320 | 530 |
| B: GD-swin-b → SAM2.1-small     | 5 | 5 | 480 | 410 | 890 |

Pipeline B provides higher-quality detection (Swin-B backbone) and more precise segmentation (SAM2.1).  
Pipeline A is suitable for latency-sensitive applications.